In [1]:
!pip install -q pyulog pandas numpy scipy matplotlib requests tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 2.3 MB/s eta 0:00:00


In [10]:
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import requests
from tqdm import tqdm
from pyulog import ULog

In [11]:
# Public PX4 flight review API root. The site's browse page and its own
# Tools/download_logs.py hit this same host.
PX4_REVIEW_BASE = "https://review.px4.io"
API_LOGS_URL = f"{PX4_REVIEW_BASE}/api/logs"      # paginated public log index
API_FACETS_URL = f"{PX4_REVIEW_BASE}/api/facets"  # filterable field values

In [12]:
LOG_DIR = "/content/px4_logs"
os.makedirs(LOG_DIR, exist_ok=True)

In [ ]:
# Pull the current, maintained download script straight from the PX4 repo.
subprocess.run(
    ["curl", "-sO",
     "https://raw.githubusercontent.com/PX4/PX4-Autopilot/main/Tools/px4tracelog/download_logs.py"],
    check=True, cwd=LOG_DIR,
)

In [3]:
# ============================================================
# STEP 0: Download logs from the PX4 public database
# ============================================================
# The official download_logs.py script depends on a local flight_review
# checkout and a DB layer that isn't Colab-friendly. Instead we hit the
# public JSON API directly (same one the browse page + script use),
# which works fine from Colab with plain `requests`.

import subprocess, sys, os

LOG_DIR = "/content/px4_logs"
os.makedirs(LOG_DIR, exist_ok=True)

# Pull the current, maintained download script straight from the PX4 repo
subprocess.run(
    ["curl", "-sO",
     "https://raw.githubusercontent.com/PX4/PX4-Autopilot/main/Tools/px4tracelog/download_logs.py"],
    check=True, cwd=LOG_DIR,
)


def fetch_log_index(max_entries=None, page_size=100):
    """Fetch metadata for public logs via the new paginated /api/logs endpoint."""
    entries = []
    offset = 0
    while True:
        resp = requests.get(
            API_LOGS_URL,
            params={"limit": page_size, "offset": offset, "is_public": True},
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()
        # response is paginated -> unwrap; adjust key if the real payload differs
        batch = data.get("logs", data) if isinstance(data, dict) else data
        if not batch:
            break
        entries.extend(batch)
        offset += len(batch)
        if max_entries and len(entries) >= max_entries:
            entries = entries[:max_entries]
            break
        if len(batch) < page_size:
            break
    return pd.DataFrame(entries)


def download_log(log_id, out_dir=LOG_DIR, overwrite=False):
    """Download a single .ulg file by its log id (new data/:filename route)."""
    out_path = os.path.join(out_dir, f"{log_id}.ulg")
    if os.path.exists(out_path) and not overwrite:
        return out_path
    url = f"{API_LOGS_URL}/{log_id}/data/{log_id}.ulg"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(out_path, "wb") as f:
        f.write(r.content)
    return out_path


def download_filtered_logs(
    n_logs=20,
    mav_type_filter=None,     # e.g. "Quadrotor"
    min_duration_s=60,        # skip very short hover-only logs
    max_duration_s=None,
    overwrite=False,
):
    """
    Pull the public log index, filter by vehicle type / duration,
    and download the matching .ulg files.
    """
    df = fetch_log_index(max_entries=None)

    # Column names on the API can shift slightly between deployments;
    # print df.columns once if this filtering silently returns nothing.
    if mav_type_filter and "mav_type" in df.columns:
        df = df[df["mav_type"].str.contains(mav_type_filter, case=False, na=False)]

    if "duration" in df.columns:
        if min_duration_s:
            df = df[df["duration"] >= min_duration_s]
        if max_duration_s:
            df = df[df["duration"] <= max_duration_s]

    df = df.head(n_logs)
    print(f"Selected {len(df)} logs matching filters.")

    paths = []
    for log_id in tqdm(df["log_id"] if "log_id" in df.columns else df["id"]):
        try:
            p = download_log(log_id, overwrite=overwrite)
            paths.append(p)
        except Exception as e:
            print(f"  skip {log_id}: {e}")
    return paths, df

In [4]:
# ============================================================
# STEP 1: Stage A — PID gains (9 values, read directly from params)
# ============================================================

PID_PARAM_MAP = {
    "Kp_x": "MC_ROLLRATE_P",  "Ki_x": "MC_ROLLRATE_I",  "Kd_x": "MC_ROLLRATE_D",
    "Kp_y": "MC_PITCHRATE_P", "Ki_y": "MC_PITCHRATE_I", "Kd_y": "MC_PITCHRATE_D",
    "Kp_z": "MC_YAWRATE_P",   "Ki_z": "MC_YAWRATE_I",   "Kd_z": "MC_YAWRATE_D",
}


def extract_pid_gains(ulog_path):
    """Read the 9 rate-controller PID gains straight from log parameters."""
    ulog = ULog(ulog_path)  # params are parsed regardless of message filter
    params = ulog.initial_parameters
    out = {}
    for name, px4_key in PID_PARAM_MAP.items():
        out[name] = params.get(px4_key, np.nan)
    out["log_file"] = os.path.basename(ulog_path)
    return out

In [5]:
# ============================================================
# STEP 2: Stage 2 — Battery / ESC model data
# ============================================================
# We need: battery voltage (U_bat), a normalized actuator command (cmd),
# and something to regress against (thrust proxy or motor current).
# battery_status gives U_bat + total current; actuator_motors / actuator_outputs
# gives per-channel commands. esc_status (if present) gives per-motor
# current/rpm/voltage which is the richest source for Eq. 6 fitting.

def extract_battery_esc_dataframe(ulog_path):
    ulog = ULog(ulog_path)
    data = {d.name: d for d in ulog.data_list}

    frames = []

    if "battery_status" in data:
        bs = pd.DataFrame(data["battery_status"].data)
        bs = bs.rename(columns={
            "timestamp": "t",
            "voltage_v": "U_bat",
            "current_a": "I_bat",
        })
        bs = bs[[c for c in ["t", "U_bat", "I_bat"] if c in bs.columns]]
        frames.append(("battery_status", bs))

    cmd_topic = None
    for candidate in ["actuator_motors", "actuator_outputs"]:
        if candidate in data:
            cmd_topic = candidate
            break

    if cmd_topic:
        ac = pd.DataFrame(data[cmd_topic].data)
        ac = ac.rename(columns={"timestamp": "t"})
        frames.append((cmd_topic, ac))

    if "esc_status" in data:
        esc = pd.DataFrame(data["esc_status"].data)
        esc = esc.rename(columns={"timestamp": "t"})
        frames.append(("esc_status", esc))

    if not frames:
        return None

    # Merge on nearest timestamp (battery/esc/actuator run at different rates)
    merged = frames[0][1].sort_values("t")
    for name, fr in frames[1:]:
        fr = fr.sort_values("t")
        merged = pd.merge_asof(merged, fr, on="t", direction="nearest",
                                suffixes=("", f"_{name}"))

    merged["log_file"] = os.path.basename(ulog_path)
    return merged


def fit_battery_esc_coeffs(df, cmd_col, omega_ss_col, c_d_global=None):
    """
    Fits Eq. 6 exactly as defined in esc_battery_model():

        Omega_ss = b1 + b2*U_bat + b3*sqrt(cmd) + b4*cmd + b5*U_bat*sqrt(cmd)

    df           : merged battery/esc dataframe (from extract_battery_esc_dataframe)
    cmd_col      : column name holding the normalized motor command (0-1),
                   e.g. an actuator_motors / actuator_outputs channel
    omega_ss_col : column name holding measured motor speed in rad/s.
                   Use esc_status motor_rpm_j * (2*pi/60), restricted to
                   quasi-steady segments (low |Omega_dot|) so it's a valid
                   stand-in for the steady-state target Omega_ss.

    eta (motor efficiency) is solved from the same model's power balance,
    P_mot = c_d * Omega^3 / eta  =>  eta = c_d * Omega^3 / P_elec,
    using P_elec = U_bat * I_bat. Requires I_bat in df and a known c_d
    (global propeller drag coefficient, usually from a bench/prop-curve
    step -- pass it in once you have it; leave None to skip eta).
    """
    d = df.dropna(subset=["U_bat", cmd_col, omega_ss_col]).copy()
    d = d[d[cmd_col] >= 0]

    U = d["U_bat"].to_numpy()
    cmd = d[cmd_col].to_numpy()
    y = d[omega_ss_col].to_numpy()

    X = np.column_stack([
        np.ones_like(U),
        U,
        np.sqrt(cmd),
        cmd,
        U * np.sqrt(cmd),
    ])

    coeffs, residuals, rank, sv = np.linalg.lstsq(X, y, rcond=None)
    b1, b2, b3, b4, b5 = coeffs

    eta = np.nan
    if c_d_global is not None and "I_bat" in d.columns:
        P_elec = (d["U_bat"] * d["I_bat"]).to_numpy()
        P_elec = np.where(np.abs(P_elec) < 1e-6, np.nan, P_elec)
        eta_samples = c_d_global * y**3 / P_elec
        eta = np.nanmedian(eta_samples)

    return {"eta": eta, "b1": b1, "b2": b2, "b3": b3, "b4": b4, "b5": b5}


In [6]:
# ============================================================
# STEP 3: Stage 7 — Aerodynamic coefficients (26 values)
# ============================================================
# Needs: body-frame velocity, body-frame specific force / torque proxy,
# and control input (actuator commands / thrust & torque setpoints),
# at a good sample rate, ideally from acro/manual flights (more excitation
# than pure hover/position-hold).

def extract_aero_dataframe(ulog_path):
    ulog = ULog(ulog_path)
    data = {d.name: d for d in ulog.data_list}
    frames = []

    if "vehicle_local_position" in data:
        vlp = pd.DataFrame(data["vehicle_local_position"].data)
        vlp = vlp.rename(columns={"timestamp": "t"})
        keep = ["t", "vx", "vy", "vz", "ax", "ay", "az"]
        vlp = vlp[[c for c in keep if c in vlp.columns]]
        frames.append(("vlp", vlp))

    if "vehicle_attitude" in data:
        att = pd.DataFrame(data["vehicle_attitude"].data)
        att = att.rename(columns={"timestamp": "t"})
        keep = ["t", "q[0]", "q[1]", "q[2]", "q[3]"]
        att = att[[c for c in keep if c in att.columns]]
        frames.append(("att", att))

    if "vehicle_angular_velocity" in data:
        av = pd.DataFrame(data["vehicle_angular_velocity"].data)
        av = av.rename(columns={"timestamp": "t"})
        keep = ["t", "xyz[0]", "xyz[1]", "xyz[2]"]
        av = av[[c for c in keep if c in av.columns]]
        frames.append(("angvel", av))

    if "vehicle_torque_setpoint" in data:
        tq = pd.DataFrame(data["vehicle_torque_setpoint"].data)
        tq = tq.rename(columns={"timestamp": "t"})
        frames.append(("torque_sp", tq))

    if "vehicle_thrust_setpoint" in data:
        th = pd.DataFrame(data["vehicle_thrust_setpoint"].data)
        th = th.rename(columns={"timestamp": "t"})
        frames.append(("thrust_sp", th))

    if not frames:
        return None

    merged = frames[0][1].sort_values("t")
    for name, fr in frames[1:]:
        fr = fr.sort_values("t")
        merged = pd.merge_asof(merged, fr, on="t", direction="nearest",
                                suffixes=("", f"_{name}"))

    merged["log_file"] = os.path.basename(ulog_path)
    return merged


def fit_aero_coeffs_generic(df, y_col, regressor_cols):
    """
    Generic least-squares fit for any of the fx/fy/fz/taux/tauy/tauz
    coefficient sets once you've defined the regressor basis functions
    for that channel (this is where Eq.-specific structure from your
    Stage 7 model goes — plug in velocity, |v|*v, angular rate products,
    control terms etc. as columns in `regressor_cols`).
    """
    d = df.dropna(subset=[y_col] + regressor_cols).copy()
    X = d[regressor_cols].to_numpy()
    y = d[y_col].to_numpy()
    coeffs, *_ = np.linalg.lstsq(X, y, rcond=None)
    return dict(zip(regressor_cols, coeffs))


In [7]:
# ============================================================
# STEP 4: Driver — run across many logs, aggregate results
# ============================================================

def run_pid_extraction(log_paths):
    rows = [extract_pid_gains(p) for p in log_paths]
    return pd.DataFrame(rows)


def run_battery_esc_extraction(log_paths):
    dfs = []
    for p in log_paths:
        d = extract_battery_esc_dataframe(p)
        if d is not None:
            dfs.append(d)
    if not dfs:
        return None
    return pd.concat(dfs, ignore_index=True)


def run_aero_extraction(log_paths):
    dfs = []
    for p in log_paths:
        d = extract_aero_dataframe(p)
        if d is not None:
            dfs.append(d)
    if not dfs:
        return None
    return pd.concat(dfs, ignore_index=True)


In [8]:
# --- run once to discover real field names ---
r = requests.get(API_FACETS_URL, timeout=30)
r.raise_for_status()
print(r.json())          # shows filterable field names/values

sample = fetch_log_index(max_entries=3)
print(sample.columns.tolist())
print(sample.head())

NameError: name 'API_FACETS_URL' is not defined

In [9]:
# --- replace download_filtered_logs() call in Cell 7 with a subprocess call ---
result = subprocess.run(
    [sys.executable, os.path.join(LOG_DIR, "download_logs.py"),
     "--mav_type", "Quadrotor",
     "--min_duration", "90",
     "--num", "15",
     "--output_dir", LOG_DIR],
    capture_output=True, text=True,
)
print(result.stdout, result.stderr)

paths = [os.path.join(LOG_DIR, f) for f in os.listdir(LOG_DIR) if f.endswith(".ulg")]
 pid_df = run_pid_extraction(paths)
 print(pid_df.describe())

 batt_df = run_battery_esc_extraction(paths)
 if batt_df is not None:
     print(batt_df.columns.tolist())

 aero_df = run_aero_extraction(paths)
 if aero_df is not None:
     print(aero_df.columns.tolist())

IndentationError: unexpected indent (977078746.py, line 13)